In [ ]:
! pip install mlflow

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Program Files\Python311\python.exe -m pip install --upgrade pip


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn

from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

# ==========================
# Load Dataset
# ==========================
digits = datasets.load_digits()

X = digits.data
y = digits.target

print("Dataset Shape:", X.shape)
print("Number of Classes:", len(np.unique(y)))

# ==========================
# Split Dataset
# ==========================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# ==========================
# Feature Scaling
# ==========================
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ==========================
# Start MLflow Run
# ==========================
with mlflow.start_run(run_name="SVM_Digits"):

    # Log Parameters
    mlflow.log_param("Model", "SVM")
    mlflow.log_param("Kernel", "rbf")
    mlflow.log_param("C", 1.0)
    mlflow.log_param("Gamma", "scale")
    mlflow.log_param("Train Size", len(X_train))
    mlflow.log_param("Test Size", len(X_test))

    # ==========================
    # Train Model
    # ==========================
    svm = SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        random_state=42
    )

    svm.fit(X_train, y_train)

    # ==========================
    # Prediction
    # ==========================
    y_pred = svm.predict(X_test)

    # ==========================
    # Evaluation
    # ==========================
    accuracy = accuracy_score(y_test, y_pred)

    print(f"\nAccuracy = {accuracy:.4f}")

    print("\nConfusion Matrix:")
    cm = confusion_matrix(y_test, y_pred)
    print(cm)

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # Log Metrics
    mlflow.log_metric("Accuracy", accuracy)

    # Log Classification Report
    report = classification_report(y_test, y_pred)
    with open("classification_report.txt", "w") as f:
        f.write(report)

    mlflow.log_artifact("classification_report.txt")

    # Log Confusion Matrix Figure
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap="Blues")

    plt.savefig("confusion_matrix.png")
    plt.close()

    mlflow.log_artifact("confusion_matrix.png")

    # Log Model
    mlflow.sklearn.log_model(
        sk_model=svm,
        artifact_path="svm_model"
    )

print("\nRun completed successfully!")

2026/07/23 19:59:20 INFO mlflow.tracking.fluent: Experiment with name 'Digits_SVM' does not exist. Creating a new experiment.


Dataset Shape: (1797, 64)
Number of Classes: 10

Accuracy = 0.9833

Confusion Matrix:
[[54  0  0  0  0  0  0  0  0  0]
 [ 0 54  0  0  1  0  0  0  0  0]
 [ 0  0 52  0  1  0  0  0  0  0]
 [ 0  0  0 55  0  0  0  0  0  0]
 [ 0  0  0  0 53  0  0  1  0  0]
 [ 0  0  0  0  0 54  0  0  0  1]
 [ 0  0  0  0  0  0 54  0  0  0]
 [ 0  0  0  0  0  0  0 54  0  0]
 [ 0  2  0  0  1  0  0  0 49  0]
 [ 0  0  0  0  0  0  1  1  0 52]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        54
           1       0.96      0.98      0.97        55
           2       1.00      0.98      0.99        53
           3       1.00      1.00      1.00        55
           4       0.95      0.98      0.96        54
           5       1.00      0.98      0.99        55
           6       0.98      1.00      0.99        54
           7       0.96      1.00      0.98        54
           8       1.00      0.94      0.97        52
           9      

2026/07/23 19:59:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Run completed successfully!


In [13]:
import joblib

joblib.dump(svm, "model.pkl")
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']